# Tutorial 4: Spectral analysis of test fixtures

The reachq test fixtures come from the algebraic graph theory
literature. Each has a known spectrum (eigenvalues of the
adjacency matrix). This notebook verifies that the generator
output matches the published spectrum.

Reference: the spectrum of the Petersen graph is {3, 1^5, -2^4}
(3, then 1 with multiplicity 5, then -2 with multiplicity 4).
The Paley graph Paley(q) for prime q ≡ 1 (mod 4) has spectrum
{ (q-1)/2, (√q-1)/2 with multiplicity (q-1)/2, (-√q-1)/2 with
multiplicity (q-1)/2 }. The Hamming graph H(d, q) has spectrum
values d-2k for k=0..d, with multiplicity C(d, k) (q-1)^k.

In [ ]:
import sys
sys.path.insert(0, '/Users/sachin/repo/parallel-reachability-and-shortest-paths')

import numpy as np

from reachq.generators import (
    hamming_graph, paley_graph, petersen_graph, shrikhande_graph
)
from reachq.spectrum import spectrum, spectral_gap

In [ ]:
# Petersen graph: spectrum = {3, 1^5, -2^4}
g = petersen_graph()
eigs = sorted(spectrum(g).tolist())
expected = sorted([-2.0]*4 + [1.0]*5 + [3.0])
print(f'Petersen: n={g.num_vertices()}, eigenvalues = {[round(e, 3) for e in eigs]}')
print(f'  expected: {expected}')
print(f'  match: {[round(e, 3) for e in eigs] == expected}')

In [ ]:
# Paley graph Paley(13): spectrum = {6, ((-1+sqrt(13))/2)^6, ((-1-sqrt(13))/2)^6}
import math
g = paley_graph(13)
eigs = sorted(spectrum(g).tolist())
expected = sorted([6.0] + [(-1+math.sqrt(13))/2]*6 + [(-1-math.sqrt(13))/2]*6)
print(f'Paley(13): n={g.num_vertices()}, eigenvalues = {[round(e, 3) for e in eigs]}')
print(f'  expected: {[round(e, 3) for e in expected]}')
print(f'  match: {all(abs(a-b) < 1e-6 for a, b in zip(sorted(eigs), sorted(expected)))}')

In [ ]:
# Shrikhande graph (4x4 rook's graph): spectrum = {6, 2^6, -2^9}
g = shrikhande_graph()
eigs = sorted(spectrum(g).tolist())
expected = sorted([6.0] + [2.0]*6 + [-2.0]*9)
print(f'Shrikhande: n={g.num_vertices()}, eigenvalues = {[round(e, 3) for e in eigs]}')
print(f'  expected: {[round(e, 3) for e in expected]}')
print(f'  match: {[round(e, 3) for e in eigs] == expected}')

In [ ]:
# Hamming graph H(2, 3): spectrum = {4, 1^4, -2^4}
g = hamming_graph(2, 3)
eigs = sorted(spectrum(g).tolist())
expected = sorted([4.0] + [1.0]*4 + [-2.0]*4)
print(f'H(2,3): n={g.num_vertices()}, eigenvalues = {[round(e, 3) for e in eigs]}')
print(f'  expected: {[round(e, 3) for e in expected]}')
print(f'  match: {[round(e, 3) for e in eigs] == expected}')

In [ ]:
# Spectral gap: largest non-trivial eigenvalue
from reachq.graph import Digraph

import matplotlib.pyplot as plt

graphs = {
    'Petersen': petersen_graph(),
    'Paley(13)': paley_graph(13),
    'Shrikhande': shrikhande_graph(),
    'Hamming(2,3)': hamming_graph(2, 3),
}
labels = []
gaps = []
for name, g in graphs.items():
    labels.append(f'{name} (n={g.num_vertices()})')
    gaps.append(spectral_gap(g))

plt.figure(figsize=(8, 4))
plt.bar(labels, gaps)
plt.ylabel('spectral gap')
plt.title('Spectral gap of reachq test fixtures')
plt.xticks(rotation=15)
plt.grid(True, axis='y')
plt.show()

## What you should see

1. The Petersen spectrum is {3, 1^5, -2^4} as expected.
2. The Paley(13) spectrum has 6 as the largest eigenvalue, with
   (√13 - 1)/2 ≈ 1.303 and -(√13 + 1)/2 ≈ -2.303 each with
   multiplicity 6.
3. The Shrikhande (rook's) graph has 6 as largest, 2 with
   multiplicity 6, and -2 with multiplicity 9.
4. The Hamming H(2, 3) graph has 4 as largest (the diameter of
   K_3 ☐ K_3 is 2, but the *spectral* gap is 4-1 = 3).
5. The spectral gap is a measure of the graph's expander
   properties: smaller gap means slower mixing.

## What's next

Tutorial 5 walks through the hopset-based SSSP and shows how
reachq compares to plain Dijkstra on the same input.